# Power Line Segmentation Prediction - Multispectral Data

This notebook performs power line segmentation prediction using multispectral imagery and trained models.

## Workflow:
- Load preprocessed 512x512 tiles from ml_ready dataset
- Use RGB + multispectral bands + vegetation indices as input
- Predict power line segmentation masks
- Evaluate performance and visualize results

## 1. Import Required Libraries

In [1]:
import os
import cv2
import torch
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from pathlib import Path
import json

print("All libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Segmentation Models PyTorch version: {smp.__version__}")

All libraries imported successfully
PyTorch version: 2.7.1
Segmentation Models PyTorch version: 0.5.0


## 2. Configuration and Hyperparameters

In [ ]:
# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

if DEVICE == "mps":
    torch.backends.mps.allow_tf32 = True
    print("MPS optimizations enabled")

# Data paths - Updated to use multispectral/data/ml_ready
DATA_DIR = Path('./data/ml_ready')
RGB_DIR = DATA_DIR / 'rgb_tiles'
BANDS_DIR = DATA_DIR / 'band_tiles'
INDEX_DIR = DATA_DIR / 'index_tiles'
MASK_DIR = DATA_DIR / 'mask_tiles'
METADATA_DIR = DATA_DIR / 'metadata'
MODEL_SAVE_DIR = Path('multispectral/models')
PREDICTION_OUTPUT_DIR = Path('multispectral/predictions')

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(PREDICTION_OUTPUT_DIR, exist_ok=True)

# Image parameters
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512
TRAIN_TEST_SPLIT_RATIO = 0.2
RANDOM_SEED = 42

# Model configuration
PL_ENCODER = "resnet34"
PL_ARCHITECTURE = "DeepLabV3"
PL_BATCH_SIZE = 16  # Increased from 8 to 16 (2x faster)
PL_NUM_EPOCHS = 3
PL_LEARNING_RATE = 0.0001
PL_WEIGHT_DECAY = 0.00001

# Loss parameters
TVERSKY_ALPHA = 0.3
TVERSKY_BETA = 0.7

# Input channel configuration
USE_MULTISPECTRAL = True  # Use multispectral bands
USE_INDICES = True  # Use vegetation indices

# Calculate input channels
# RGB: 3 channels
# Multispectral bands: 4 (green, red, rededge, nir)
# Vegetation indices: 5 (NDVI, GNDVI, NDRE, LCI, OSAVI)
INPUT_CHANNELS = 3  # RGB
if USE_MULTISPECTRAL:
    INPUT_CHANNELS += 4
if USE_INDICES:
    INPUT_CHANNELS += 5

print(f"\nPower Line Model Configuration:")
print(f"{PL_ARCHITECTURE}+{PL_ENCODER}, Batch={PL_BATCH_SIZE}, Epochs={PL_NUM_EPOCHS}")
print(f"Input Channels: {INPUT_CHANNELS} (RGB: 3, Bands: {4 if USE_MULTISPECTRAL else 0}, Indices: {5 if USE_INDICES else 0})")
print(f"Image Size: {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
print(f"Tversky Loss: α={TVERSKY_ALPHA}, β={TVERSKY_BETA}")
print(f"\nData paths:")
print(f"  Data directory: {DATA_DIR}")
print(f"  Model save directory: {MODEL_SAVE_DIR}")
print(f"\nSpeed optimizations:")
print(f"  - Batch size: {PL_BATCH_SIZE} (larger = faster)")
print(f"  - MPS optimizations: {'Enabled' if DEVICE == 'mps' else 'N/A'}")

Using device: mps
MPS optimizations enabled

Power Line Model Configuration:
DeepLabV3+resnet34, Batch=16, Epochs=25
Input Channels: 12 (RGB: 3, Bands: 4, Indices: 5)
Image Size: 512x512
Tversky Loss: α=0.3, β=0.7

Data paths:
  Data directory: data/ml_ready
  Model save directory: multispectral/models

Speed optimizations:
  - Batch size: 16 (larger = faster)
  - MPS optimizations: Enabled


## 3. Custom Loss Functions

In [3]:
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, smooth=1e-6):
        super(TverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        TP = (inputs * targets).sum()    
        FP = ((1-targets) * inputs).sum()
        FN = (targets * (1-inputs)).sum()
       
        denominator = TP + self.alpha*FP + self.beta*FN + self.smooth
        Tversky = (TP + self.smooth) / denominator
        focal_weight = torch.pow(1 - Tversky, 2)
        
        return focal_weight * (1 - Tversky)

class CombinedLoss(nn.Module):
    def __init__(self, tversky_weight=0.7, bce_weight=0.3, pos_weight=10.0):
        super(CombinedLoss, self).__init__()
        self.tversky_weight = tversky_weight
        self.bce_weight = bce_weight
        self.tversky_loss = TverskyLoss(alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA)
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
    
    def forward(self, inputs, targets):
        tversky = self.tversky_loss(inputs, targets)
        bce = self.bce_loss(inputs, targets)
        return self.tversky_weight * tversky + self.bce_weight * bce

print("Loss functions defined successfully")
print(f"Combined loss: 70% Tversky + 30% Weighted BCE")

Loss functions defined successfully
Combined loss: 70% Tversky + 30% Weighted BCE


## 4. Custom Dataset for Multispectral Power Line Segmentation

In [4]:
class MultispectralPowerLineDataset(Dataset):
    """
    Dataset for power line segmentation using multispectral data.
    Loads RGB, multispectral bands, vegetation indices, and power line masks.
    """
    def __init__(self, tile_names, rgb_dir, bands_dir, index_dir, mask_dir, 
                 transform=None, use_multispectral=True, use_indices=True):
        self.tile_names = tile_names
        self.rgb_dir = Path(rgb_dir)
        self.bands_dir = Path(bands_dir)
        self.index_dir = Path(index_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform
        self.use_multispectral = use_multispectral
        self.use_indices = use_indices

    def __len__(self):
        return len(self.tile_names)

    def __getitem__(self, idx):
        tile_name = self.tile_names[idx]
        
        # Load RGB image
        rgb_path = self.rgb_dir / f"{tile_name}.png"
        rgb_image = cv2.imread(str(rgb_path))
        rgb_image = cv2.cvtColor(rgb_image, cv2.COLOR_BGR2RGB)
        
        # Load power line mask
        mask_path = self.mask_dir / f"{tile_name}_powerline.npy"
        mask = np.load(mask_path).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)
        
        # Initialize combined image with RGB
        combined_image = rgb_image.copy()
        
        # Load and concatenate multispectral bands
        if self.use_multispectral:
            bands = ['green', 'red', 'rededge', 'nir']
            band_data = []
            for band in bands:
                band_path = self.bands_dir / f"{tile_name}_{band}.npy"
                band_array = np.load(band_path)
                
                # Handle NaN and inf values
                band_array = np.nan_to_num(band_array, nan=0.0, posinf=0.0, neginf=0.0)
                
                # Normalize band to 0-255 range with proper handling
                band_min = band_array.min()
                band_max = band_array.max()
                
                if band_max - band_min > 1e-8:
                    band_normalized = ((band_array - band_min) / (band_max - band_min) * 255).astype(np.uint8)
                else:
                    # Uniform band - set to mid-range
                    band_normalized = np.full_like(band_array, 128, dtype=np.uint8)
                
                band_data.append(band_normalized)
            
            # Stack bands
            bands_stacked = np.stack(band_data, axis=-1)
            combined_image = np.concatenate([combined_image, bands_stacked], axis=-1)
        
        # Load and concatenate vegetation indices
        if self.use_indices:
            indices = ['ndvi', 'gndvi', 'ndre', 'lci', 'osavi']
            index_data = []
            for index in indices:
                index_path = self.index_dir / f"{tile_name}_{index}.npy"
                index_array = np.load(index_path)
                
                # Handle NaN and inf values
                index_array = np.nan_to_num(index_array, nan=0.0, posinf=1.0, neginf=-1.0)
                
                # Clip to valid range for vegetation indices (-1 to 1)
                index_array = np.clip(index_array, -1, 1)
                
                # Normalize index to 0-255 range
                index_normalized = ((index_array + 1) / 2 * 255).astype(np.uint8)
                index_data.append(index_normalized)
            
            # Stack indices
            indices_stacked = np.stack(index_data, axis=-1)
            combined_image = np.concatenate([combined_image, indices_stacked], axis=-1)
        
        # Apply transformations
        if self.transform:
            augmented = self.transform(image=combined_image, mask=mask)
            combined_image = augmented['image']
            mask = augmented['mask']
            mask = mask.permute(2, 0, 1)

        return combined_image, mask

print("MultispectralPowerLineDataset class defined successfully")
print("Note: Handles NaN/inf values in bands and indices")

MultispectralPowerLineDataset class defined successfully
Note: Handles NaN/inf values in bands and indices


## 5. Training and Evaluation Functions

In [5]:
def train_fn(loader, model, optimizer, loss_fn, scaler=None):
    model.train()
    loop = tqdm(loader, desc="Training Power Line Model")
    total_loss = 0.0
    num_batches = len(loader)
    
    use_amp = (DEVICE == "cuda")
    
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE, non_blocking=True)
        targets = targets.float().to(device=DEVICE, non_blocking=True)
        
        if data.size(0) < 2:
            continue

        if use_amp and scaler is not None:
            with torch.autocast(device_type="cuda", enabled=True):
                predictions = model(data)
                loss = loss_fn(predictions, targets)
        else:
            predictions = model(data)
            loss = loss_fn(predictions, targets)
        
        total_loss += loss.item()

        optimizer.zero_grad()
        
        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        
        # Reduced MPS cache clearing frequency (only every 100 batches instead of 50)
        if DEVICE == "mps" and batch_idx % 100 == 0:
            torch.mps.empty_cache()

        loop.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg_loss': f'{total_loss/(batch_idx+1):.4f}'
        })
        
    return total_loss / num_batches


def check_accuracy(loader, model, device):
    model.eval()
    num_correct = 0
    num_pixels = 0
    dice_score = 0
    iou_score = 0
    valid_batches = 0

    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(loader):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            
            if x.size(0) < 1:
                continue
            
            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
            
            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)
            
            tp, fp, fn, tn = smp.metrics.get_stats(preds.long(), y.long(), mode='binary')
            batch_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
            batch_dice = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")
            
            if not torch.isnan(batch_iou):
                iou_score += batch_iou
                valid_batches += 1
            if not torch.isnan(batch_dice):
                dice_score += batch_dice
            
            # Reduced MPS cache clearing frequency
            if DEVICE == "mps" and batch_idx % 100 == 0:
                torch.mps.empty_cache()

    if valid_batches > 0:
        iou = iou_score / valid_batches
    else:
        iou = torch.tensor(0.0)
    
    dice = dice_score / len(loader)
    acc = num_correct / num_pixels * 100
    
    print(f"Validation: IoU Score: {iou:.4f} | Dice Score: {dice:.4f} | Pixel Accuracy: {acc:.2f}%")

    model.train()
    return iou.item()

print("Training and evaluation functions defined successfully")
print("Optimization: Reduced MPS cache clearing frequency (every 100 batches)")

Training and evaluation functions defined successfully
Optimization: Reduced MPS cache clearing frequency (every 100 batches)


## 6. Data Loading and Preparation

In [6]:
print("Loading tile metadata...")

# Load metadata
metadata_path = METADATA_DIR / 'tile_metadata.json'
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print(f"Total tiles in dataset: {metadata['total_tiles']}")
print(f"Tile size: {metadata['tile_size']}x{metadata['tile_size']}")

# Get all tile names
all_tile_names = [tile['tile_name'] for tile in metadata['tiles']]

# Separate tiles by power line presence for stratified splitting
tiles_with_powerlines = [tile['tile_name'] for tile in metadata['tiles'] if tile.get('has_power_line', False)]
tiles_without_powerlines = [tile['tile_name'] for tile in metadata['tiles'] if not tile.get('has_power_line', False)]

print(f"\nTotal tiles: {len(all_tile_names)}")
print(f"Tiles with power lines: {len(tiles_with_powerlines)}")
print(f"Tiles without power lines: {len(tiles_without_powerlines)}")

# Use all tiles for training
USE_ALL_TILES = True

if USE_ALL_TILES:
    print(f"\nUsing all {len(all_tile_names)} tiles for training")
    print("Applying STRATIFIED SPLIT to ensure power line representation in all sets")
    
    # Stratified split: Split power line tiles and non-power line tiles separately
    # This ensures both train/val/test have power line samples
    
    # Split tiles WITH power lines (60% train, 20% val, 20% test)
    pl_train_val, pl_test = train_test_split(
        tiles_with_powerlines,
        test_size=0.2,
        random_state=RANDOM_SEED
    )
    pl_train, pl_val = train_test_split(
        pl_train_val,
        test_size=0.25,  # 0.25 * 0.8 = 0.2 of total
        random_state=RANDOM_SEED
    )
    
    # Split tiles WITHOUT power lines (60% train, 20% val, 20% test)
    no_pl_train_val, no_pl_test = train_test_split(
        tiles_without_powerlines,
        test_size=0.2,
        random_state=RANDOM_SEED
    )
    no_pl_train, no_pl_val = train_test_split(
        no_pl_train_val,
        test_size=0.25,
        random_state=RANDOM_SEED
    )
    
    # Combine stratified splits
    train_tiles = pl_train + no_pl_train
    val_tiles = pl_val + no_pl_val
    test_tiles = pl_test + no_pl_test
    
    # Shuffle combined lists
    import random
    random.seed(RANDOM_SEED)
    random.shuffle(train_tiles)
    random.shuffle(val_tiles)
    random.shuffle(test_tiles)
    
    tile_names = all_tile_names
    
else:
    # Use only tiles with power lines
    tile_names = tiles_with_powerlines
    print(f"\nUsing only {len(tile_names)} tiles with power lines")
    
    # Standard split for power line tiles only
    train_val_tiles, test_tiles = train_test_split(
        tile_names, 
        test_size=0.2, 
        random_state=RANDOM_SEED
    )
    train_tiles, val_tiles = train_test_split(
        train_val_tiles,
        test_size=0.25,
        random_state=RANDOM_SEED
    )

# Count power line tiles in each set
train_pl_count = sum(1 for t in train_tiles if t in tiles_with_powerlines)
val_pl_count = sum(1 for t in val_tiles if t in tiles_with_powerlines)
test_pl_count = sum(1 for t in test_tiles if t in tiles_with_powerlines)

print(f"\nStratified data split completed:")
print(f"Training samples: {len(train_tiles)} ({len(train_tiles)/len(tile_names)*100:.1f}%) - {train_pl_count} with power lines")
print(f"Validation samples: {len(val_tiles)} ({len(val_tiles)/len(tile_names)*100:.1f}%) - {val_pl_count} with power lines")
print(f"Test samples: {len(test_tiles)} ({len(test_tiles)/len(tile_names)*100:.1f}%) - {test_pl_count} with power lines")

print(f"\nPower line representation:")
print(f"Training: {train_pl_count}/{len(train_tiles)} ({train_pl_count/len(train_tiles)*100:.1f}%)")
print(f"Validation: {val_pl_count}/{len(val_tiles)} ({val_pl_count/len(val_tiles)*100:.1f}%)")
print(f"Test: {test_pl_count}/{len(test_tiles)} ({test_pl_count/len(test_tiles)*100:.1f}%)")

Loading tile metadata...
Total tiles in dataset: 702
Tile size: 512x512

Total tiles: 702
Tiles with power lines: 109
Tiles without power lines: 593

Using all 702 tiles for training
Applying STRATIFIED SPLIT to ensure power line representation in all sets

Stratified data split completed:
Training samples: 420 (59.8%) - 65 with power lines
Validation samples: 141 (20.1%) - 22 with power lines
Test samples: 141 (20.1%) - 22 with power lines

Power line representation:
Training: 65/420 (15.5%)
Validation: 22/141 (15.6%)
Test: 22/141 (15.6%)


## 7. Data Augmentation

In [7]:
print("Defining data augmentation...")

# Training augmentation (geometric only - no color transforms for multispectral data)
train_transform = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=15, p=0.5),
    # Note: Skip ColorJitter for multispectral data (>3 channels)
    A.Normalize(mean=[0.485]*INPUT_CHANNELS, std=[0.229]*INPUT_CHANNELS, max_pixel_value=255.0),
    ToTensorV2(),
])

# Validation transform (no augmentation)
val_transform = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
    A.Normalize(mean=[0.485]*INPUT_CHANNELS, std=[0.229]*INPUT_CHANNELS, max_pixel_value=255.0),
    ToTensorV2(),
])

print("Augmentation pipeline defined for multispectral data")
print("Note: Using geometric augmentations only (flip, rotate)")
print("ColorJitter skipped - not compatible with multispectral channels")

Defining data augmentation...
Augmentation pipeline defined for multispectral data
Note: Using geometric augmentations only (flip, rotate)
ColorJitter skipped - not compatible with multispectral channels


## 8. Initialize Model and Training Components

In [8]:
print("\n" + "="*70)
print("MULTISPECTRAL POWER LINE SEGMENTATION MODEL")
print("="*70)

# Initialize model
print(f"Initializing {PL_ARCHITECTURE} model with {PL_ENCODER} encoder...")
print(f"Input channels: {INPUT_CHANNELS}")

model = smp.DeepLabV3(
    encoder_name=PL_ENCODER,
    encoder_weights=None,  # No pretrained weights for custom input channels
    in_channels=INPUT_CHANNELS,
    classes=1,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,} total, {trainable_params:,} trainable")

# Initialize loss and optimizer
loss_fn = CombinedLoss(tversky_weight=0.7, bce_weight=0.3, pos_weight=15.0)

if DEVICE == "mps":
    optimizer = optim.AdamW(model.parameters(), lr=PL_LEARNING_RATE, weight_decay=PL_WEIGHT_DECAY)
    print("Using AdamW optimizer")
else:
    optimizer = optim.Adam(model.parameters(), lr=PL_LEARNING_RATE, weight_decay=PL_WEIGHT_DECAY)
    print("Using Adam optimizer")

# Learning rate scheduler
from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='max',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)
print("Learning Rate Scheduler: ReduceLROnPlateau")

# Gradient scaler for CUDA
scaler = None
if DEVICE == "cuda":
    scaler = torch.cuda.amp.GradScaler()
    print("CUDA AMP enabled")

# Create data loaders with optimizations
print("\nCreating optimized data loaders...")

# Optimized DataLoader settings for speed
num_workers = 4 if DEVICE != "mps" else 0  # Parallel data loading (MPS needs 0)
pin_memory = True if DEVICE == "cuda" else False  # Faster CPU-GPU transfer
persistent_workers = True if num_workers > 0 else False  # Keep workers alive

train_loader = DataLoader(
    MultispectralPowerLineDataset(train_tiles, RGB_DIR, BANDS_DIR, INDEX_DIR, MASK_DIR, 
                                 transform=train_transform, use_multispectral=USE_MULTISPECTRAL, 
                                 use_indices=USE_INDICES),
    batch_size=PL_BATCH_SIZE, 
    shuffle=True, 
    num_workers=num_workers, 
    drop_last=True,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=2 if num_workers > 0 else None  # Prefetch batches
)

val_loader = DataLoader(
    MultispectralPowerLineDataset(val_tiles, RGB_DIR, BANDS_DIR, INDEX_DIR, MASK_DIR, 
                                 transform=val_transform, use_multispectral=USE_MULTISPECTRAL, 
                                 use_indices=USE_INDICES),
    batch_size=PL_BATCH_SIZE, 
    shuffle=False, 
    num_workers=num_workers, 
    drop_last=False,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=2 if num_workers > 0 else None
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"\nDataLoader optimizations:")
print(f"  - num_workers: {num_workers} (parallel data loading)")
print(f"  - pin_memory: {pin_memory} (faster transfers)")
print(f"  - persistent_workers: {persistent_workers} (reuse workers)")
print(f"  - prefetch_factor: {2 if num_workers > 0 else None} (pre-load batches)")

if DEVICE == "mps":
    torch.mps.empty_cache()

print(f"\nReady to train for up to {PL_NUM_EPOCHS} epochs")


MULTISPECTRAL POWER LINE SEGMENTATION MODEL
Initializing DeepLabV3 model with resnet34 encoder...
Input channels: 12
Model parameters: 26,035,329 total, 26,035,329 trainable
Using AdamW optimizer
Learning Rate Scheduler: ReduceLROnPlateau

Creating optimized data loaders...
Training batches: 26
Validation batches: 9

DataLoader optimizations:
  - num_workers: 0 (parallel data loading)
  - pin_memory: False (faster transfers)
  - persistent_workers: False (reuse workers)
  - prefetch_factor: None (pre-load batches)

Ready to train for up to 25 epochs


## 9. Training Loop

In [ ]:
best_iou = -1
training_history = {
    'train_loss': [],
    'val_iou': [],
    'learning_rates': []
}

print("Starting training...\n")

for epoch in range(PL_NUM_EPOCHS):
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{PL_NUM_EPOCHS} | LR: {current_lr:.2e}")
    
    if DEVICE == "mps":
        torch.mps.empty_cache()
    
    # Training
    train_loss = train_fn(train_loader, model, optimizer, loss_fn, scaler)
    
    # Validation
    current_iou = check_accuracy(val_loader, model, device=DEVICE)
    
    # Update scheduler
    scheduler.step(current_iou)
    
    # Save history
    training_history['train_loss'].append(train_loss)
    training_history['val_iou'].append(current_iou)
    training_history['learning_rates'].append(current_lr)
    
    # Check for LR reduction
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"Learning rate reduced from {current_lr:.2e} to {new_lr:.2e}")
    
    # Save best model
    if current_iou > best_iou:
        best_iou = current_iou
        checkpoint = {"state_dict": model.state_dict()}
        model_path = MODEL_SAVE_DIR / "multispectral_powerline_model.pth.tar"
        torch.save(checkpoint, model_path)
        print(f"Train Loss: {train_loss:.4f} | Val IoU: {current_iou:.4f} | BEST MODEL SAVED")
    else:
        print(f"Train Loss: {train_loss:.4f} | Val IoU: {current_iou:.4f} | Best: {best_iou:.4f}")
    
    if DEVICE == "mps":
        torch.mps.empty_cache()
        
    print("-" * 80)

print(f"\nTraining completed!")
print(f"Best IoU achieved: {best_iou:.4f}")
print(f"Model saved as: multispectral_powerline_model.pth.tar")

Starting training...

Epoch 1/25 | LR: 1.00e-04


Training Power Line Model:   0%|          | 0/26 [00:00<?, ?it/s]

## 10. Testing on Unseen Data

In [ ]:
print("\n" + "="*70)
print("TESTING ON UNSEEN DATA")
print("="*70)

# Create test loader
test_loader = DataLoader(
    MultispectralPowerLineDataset(test_tiles, RGB_DIR, BANDS_DIR, INDEX_DIR, MASK_DIR, 
                                 transform=val_transform, use_multispectral=USE_MULTISPECTRAL, 
                                 use_indices=USE_INDICES),
    batch_size=PL_BATCH_SIZE, 
    shuffle=False, 
    num_workers=0
)

print(f"Test samples: {len(test_tiles)}")
print(f"Test batches: {len(test_loader)}")

# Load best model
print("Loading best model...")
checkpoint = torch.load(MODEL_SAVE_DIR / "multispectral_powerline_model.pth.tar", map_location=DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# Test
print("Testing...")
test_iou = check_accuracy(test_loader, model, device=DEVICE)

print(f"\nFINAL TEST RESULT:")
print(f"Power Line Test IoU: {test_iou:.4f}")
print("="*70)

## 11. Visualization

In [ ]:
import random

def visualize_predictions(model, test_loader, num_samples=4, device="cpu"):
    model.eval()
    
    # Get all data
    all_images = []
    all_masks = []
    
    with torch.no_grad():
        for images, masks in test_loader:
            for i in range(images.shape[0]):
                all_images.append(images[i])
                all_masks.append(masks[i])
    
    # Random samples
    random_indices = random.sample(range(len(all_images)), min(num_samples, len(all_images)))
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for sample_idx, data_idx in enumerate(random_indices):
        image = all_images[data_idx].to(device).unsqueeze(0)
        mask = all_masks[data_idx].to(device).unsqueeze(0)
        
        # Prediction
        prediction = model(image)
        prediction_prob = torch.sigmoid(prediction)
        predicted_mask = (prediction_prob > 0.5).float()
        
        # Convert to numpy (use only RGB channels for visualization)
        image_np = image.squeeze().cpu().detach()[:3].permute(1, 2, 0).numpy()
        true_mask_np = mask.squeeze().cpu().detach().numpy()
        pred_mask_np = predicted_mask.squeeze().detach().cpu().numpy()
        prob_mask_np = prediction_prob.squeeze().detach().cpu().numpy()
        
        # Denormalize RGB
        image_np = np.clip(image_np, 0, 1)
        
        # Plot
        axes[sample_idx, 0].imshow(image_np)
        axes[sample_idx, 0].set_title(f'RGB Image {sample_idx+1}')
        axes[sample_idx, 0].axis('off')
        
        axes[sample_idx, 1].imshow(true_mask_np, cmap='gray', vmin=0, vmax=1)
        axes[sample_idx, 1].set_title('Ground Truth')
        axes[sample_idx, 1].axis('off')
        
        axes[sample_idx, 2].imshow(prob_mask_np, cmap='gray', vmin=0, vmax=1)
        axes[sample_idx, 2].set_title('Prediction Probability')
        axes[sample_idx, 2].axis('off')
        
        axes[sample_idx, 3].imshow(pred_mask_np, cmap='gray', vmin=0, vmax=1)
        axes[sample_idx, 3].set_title('Binary Prediction')
        axes[sample_idx, 3].axis('off')
    
    plt.tight_layout()
    plt.suptitle('Multispectral Power Line Segmentation Results', fontsize=16, y=0.98)
    plt.savefig(PREDICTION_OUTPUT_DIR / 'predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Visualized {len(random_indices)} samples")

# Visualize
visualize_predictions(model, test_loader, num_samples=4, device=DEVICE)

## 12. Save Training History

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

epochs = range(1, len(training_history['train_loss']) + 1)

ax1.plot(epochs, training_history['train_loss'], 'b-', linewidth=2)
ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, training_history['val_iou'], 'b-', linewidth=2)
ax2.set_title('Validation IoU', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('IoU Score')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PREDICTION_OUTPUT_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTRAINING SUMMARY")
print("="*50)
print(f"Final Training Loss: {training_history['train_loss'][-1]:.4f}")
print(f"Best Validation IoU: {max(training_history['val_iou']):.4f}")
print(f"Final Validation IoU: {training_history['val_iou'][-1]:.4f}")
print(f"Test IoU: {test_iou:.4f}")
print(f"Total Epochs: {len(training_history['train_loss'])}")